# geo1242 – NDVI in code (short version, self-study)

**Unit:** geo1240 Spectral indices · **Slides:** `geo1220_RS_Fundamentals` (part *Indices*) · **Data:** `data/rs_kamp_lintfort/` (Sentinel-2 clip of 25 June 2026, ESA WorldCover) · **Builds on:** geo1241 (NDVI with the Raster Calculator in QGIS) · **Time:** about 30 min

In QGIS you computed the NDVI with the Raster Calculator. This notebook does the same in Python, with the same file. You do not have to write code: **read each text cell, predict what will happen, then run the code cell** (`Shift+Enter`) and compare with your QGIS result.

You will see how to
1. read two bands of a GeoTIFF with `rasterio` and turn the stored values into reflectance;
2. compute a normalized difference index with NumPy, with `NaN` where there is no data;
3. write the result as a GeoTIFF that QGIS can open;
4. compute the mean NDVI per land-cover class, like *Raster layer zonal statistics* in QGIS.

Open it on the JupyterHub: the data are in your session folder.

## 0. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio as rio

# released with unit geo1220; the second path is for the teacher's repo (test runs in the unit folder)
DATA = next((p for p in (Path("data/rs_kamp_lintfort"),
                         Path("../geo1220_RS_Fundamentals/data/rs_kamp_lintfort")) if p.exists()), None)
assert DATA, "data/rs_kamp_lintfort/ not found: copy it from your session material"
OUT = Path("geo1242_out")
OUT.mkdir(exist_ok=True)

S2 = DATA / "s2_kamp_lintfort.tif"           # bands 1-6: B02 B03 B04 B05 B08 B11, value = reflectance x 10 000
WORLDCOVER = {10: "tree cover", 30: "grassland", 40: "cropland", 50: "built-up", 80: "water"}

## 1. NDVI from our file

### 1.1 Read two bands
**Predict:** which band numbers of our file hold red (B04) and near infrared (B08)? Look at the band list in the header above, or at the band names QGIS showed you in geo1221.

In [ ]:
src = rio.open(S2)
print(src.descriptions)
print(src.crs, src.res, src.shape, src.dtypes[0], "nodata:", src.nodata)
red, nir = src.read((3, 5)).astype("float32") / 10000    # reflectance 0 ... 1
print(red.mean().round(3), nir.mean().round(3))

A **normalized difference** of two bands a, b is $(a - b) / (a + b)$. NDVI = nd(NIR, red). Where both bands are 0 (no data), the result is 0/0: it must become `NaN` (not a number), not 0 and not an error.

In [ ]:
def nd(a, b):
    "Normalized difference (a - b) / (a + b); NaN where a + b == 0."
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(a + b == 0, np.nan, (a - b) / (a + b)).astype("float32")

In [ ]:
ndvi = nd(nir, red)
print("NDVI min/max:", np.nanmin(ndvi).round(2), np.nanmax(ndvi).round(2))
print(f"share of pixels above 0.6: {np.nanmean(ndvi > 0.6):.1%}")

fig, ax = plt.subplots(figsize=(11, 6.5))
b = src.bounds
im = ax.imshow(ndvi, cmap="RdYlGn", vmin=-1, vmax=1, extent=(b.left, b.right, b.bottom, b.top))
fig.colorbar(im, ax=ax, label="NDVI", shrink=0.8)
ax.set_title("NDVI, Sentinel-2, 25 June 2026")
ax.ticklabel_format(style="plain")
plt.show()

**Predict before you run:** does dividing by 10 000 change the NDVI? Try `nd(nir * 10000, red * 10000)` in a new cell.

### 1.2 Write a GeoTIFF
The output keeps the grid of the input (CRS, transform, size). Only the number of bands, the data type and the no-data value change. With `float32` the natural no-data value is `NaN`.

In [ ]:
profile = src.profile | {"count": 1, "dtype": "float32", "nodata": np.nan}
with rio.open(OUT / "ndvi_20260625.tif", "w", **profile) as dst:
    dst.write(ndvi, 1)
    dst.set_band_description(1, "NDVI 2026-06-25")
print(OUT / "ndvi_20260625.tif")

Open the file in QGIS next to your Raster Calculator result from geo1241: same values?

### 1.3 NDVI per land-cover class
ESA WorldCover 2021 is an existing land-cover map on the same 10 m grid (a model result for another year: a reference, not the truth). A boolean mask `wc == 10` selects all tree-cover pixels.

**Predict:** rank the five classes by mean NDVI.

In [ ]:
wc = rio.open(DATA / "worldcover_kamp_lintfort.tif").read(1)
pd.Series({name: np.nanmean(ndvi[wc == c]) for c, name in WORLDCOVER.items()}, name="mean NDVI").round(2)

## What next?
Nothing to upload: this notebook is self-study.

* Open `geo1242_out/ndvi_20260625.tif` in QGIS next to your `ndvi.tif` from geo1241. Same values? (*Identify* one pixel in both.)
* Compare the table of §1.3 with your *Raster layer zonal statistics* table from geo1241, task 2.
* Try it yourself: add a cell and compute the NDRE = nd(B08, B05) with band 4 (B05) instead of band 3. Which class has the highest mean NDRE?
* A longer version of this notebook (searching Sentinel-2 scenes yourself, reading only your window from the cloud, cloud masks, the NDVI over a whole season) exists at M.Sc. level: ask your teacher if you want it.

Sources: contains modified Copernicus Sentinel data 2026; ESA WorldCover 2021 v200, CC BY 4.0.